In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import warnings
warnings.filterwarnings("ignore")

from tensorflow import keras
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Conv2D, MaxPooling2D , Dense , Flatten , Dropout
from tensorflow.keras.preprocessing.image import ImageDataGenerator
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.preprocessing import image

from tensorflow.keras.callbacks import EarlyStopping , ModelCheckpoint
from tensorflow.keras.layers import BatchNormalization

In [2]:
train_dir='data/'

In [3]:
train_gen=ImageDataGenerator(
        rescale=1./255,
        validation_split=0.2,
        rotation_range=20,
        zoom_range=0.2,
        horizontal_flip=True
)

train_data=train_gen.flow_from_directory(
    train_dir,
    target_size=(128,128),
    batch_size=32,
    class_mode='categorical',
    subset='training'
)


val_data = train_gen.flow_from_directory(
    train_dir,
    target_size= (128, 128) ,  
    batch_size=32,
    class_mode='categorical',
    subset='validation'      
)

Found 11706 images belonging to 3 classes.
Found 2924 images belonging to 3 classes.


In [4]:

model=Sequential([
    Conv2D(32 ,(3,3),activation='relu',input_shape=(128,128,3)),
    BatchNormalization(),
    MaxPooling2D(2,2),

    Conv2D(64,(3,3),activation='relu',input_shape=(128,128,3)),
    BatchNormalization(),
    MaxPooling2D(2,2),

    Conv2D(128,(3,3),activation='relu',input_shape=(128,128,3)),
    BatchNormalization(),
    MaxPooling2D(2,2),

    Flatten(),
    Dense(128, activation='relu'),
    Dropout(0.5),

    Dense(3, activation='softmax')

])

In [5]:
model.compile(
    optimizer = Adam( learning_rate=0.001 ),  
    loss='categorical_crossentropy',               
    metrics=['accuracy']                      
)

In [6]:
early_stop = EarlyStopping(
    monitor = 'val_loss',           
    patience = 3,                   
    restore_best_weights = True     
)


checkpoint = ModelCheckpoint(
    "best_model.keras",            
    monitor = 'val_accuracy',       
    save_best_only = True          
)

In [7]:
history = model.fit(
    train_data,                          
    epochs = 5,                           
    validation_data = val_data,            

    steps_per_epoch = len(train_data),    
    validation_steps = len(val_data),      

    callbacks = [early_stop, checkpoint]   
)

Epoch 1/5
366/366 ━━━━━━━━━━━━━━━━━━━━ 675s 2s/step - accuracy: 0.7682 - loss: 1.0271 - val_accuracy: 0.7880 - val_loss: 0.5436
Epoch 2/5
366/366 ━━━━━━━━━━━━━━━━━━━━ 659s 2s/step - accuracy: 0.8793 - loss: 0.3457 - val_accuracy: 0.9138 - val_loss: 0.2404
Epoch 3/5
366/366 ━━━━━━━━━━━━━━━━━━━━ 877s 2s/step - accuracy: 0.9185 - loss: 0.2296 - val_accuracy: 0.8307 - val_loss: 0.4092
Epoch 4/5
366/366 ━━━━━━━━━━━━━━━━━━━━ 923s 3s/step - accuracy: 0.9328 - loss: 0.1916 - val_accuracy: 0.9251 - val_loss: 0.2000
Epoch 5/5
366/366 ━━━━━━━━━━━━━━━━━━━━ 1015s 3s/step - accuracy: 0.9425 - loss: 0.1705 - val_accuracy: 0.9101 - val_loss: 0.2527


In [ ]:
plt.figure(figsize=(10,4))

# Plot accuracy
plt.subplot(1,2,1)
plt.plot(history.history['accuracy'], label='Train Accuracy')       
plt.plot(history.history['val_accuracy'], label='Val Accuracy')    
plt.title("Accuracy")
plt.legend()

# Plot loss
plt.subplot(1,2,2)
plt.plot(history.history['loss'], label='Train Loss')              
plt.plot(history.history['val_loss'], label='Val Loss')             
plt.title("Loss")
plt.legend()

plt.show()

In [8]:
img_path = 'Dog.jpg'  # Path to test image

# Load and resize image
img = image.load_img(img_path, target_size=(128, 128))

img_array = image.img_to_array(img) / 255.0

img_array = np.expand_dims(img_array, axis=0)

# Predict probability
prediction = model.predict(img_array)

print("Raw Prediction:", prediction)

class_names = ['Cat', 'Dog','Wild Animal']

for i, cls in enumerate(class_names):
    print(f"{cls}: {prediction[0][i] * 100:.2f}%")

1/1 ━━━━━━━━━━━━━━━━━━━━ 1s 506ms/step
Raw Prediction: [[5.2498752e-11 9.9999917e-01 8.5909761e-07]]
Cat: 0.00%
Dog: 100.00%
Wild Animal: 0.00%


In [9]:
# Save trained model to file
model.save("animal_final.keras")

print("Model saved successfully!")

Model saved successfully!


In [10]:
 import tensorflow as tf
from tensorflow.keras.models import load_model
converter=tf.lite.TFLiteConverter.from_keras_model(model)
converter.optimization=[tf.lite.Optimize.DEFAULT]
tflite_model=converter.convert()
with open("animal_final.tflite","wb") as f:
    f.write(tflite_model)
import os 
size=os.path.getsize("animal_final.tflite")/(1024*1024)
print(size)

INFO:tensorflow:Assets written to: C:\Users\Shravani\AppData\Local\Temp\tmpx462nzpa\assets


INFO:tensorflow:Assets written to: C:\Users\Shravani\AppData\Local\Temp\tmpx462nzpa\assets


Saved artifact at 'C:\Users\Shravani\AppData\Local\Temp\tmpx462nzpa'. The following endpoints are available:

* Endpoint 'serve'
  args_0 (POSITIONAL_ONLY): TensorSpec(shape=(None, 128, 128, 3), dtype=tf.float32, name='keras_tensor')
Output Type:
  TensorSpec(shape=(None, 3), dtype=tf.float32, name=None)
Captures:
  2893470891552: TensorSpec(shape=(), dtype=tf.resource, name=None)
  2893470964672: TensorSpec(shape=(), dtype=tf.resource, name=None)
  2893470960800: TensorSpec(shape=(), dtype=tf.resource, name=None)
  2893470968368: TensorSpec(shape=(), dtype=tf.resource, name=None)
  2893470966256: TensorSpec(shape=(), dtype=tf.resource, name=None)
  2893470967136: TensorSpec(shape=(), dtype=tf.resource, name=None)
  2893470971184: TensorSpec(shape=(), dtype=tf.resource, name=None)
  2893470969248: TensorSpec(shape=(), dtype=tf.resource, name=None)
  2893470967840: TensorSpec(shape=(), dtype=tf.resource, name=None)
  2893470971008: TensorSpec(shape=(), dtype=tf.resource, name=None)
  28